## Running This Notebook

You'll need the following libraries for this notebook:
* [Pandas](http://pandas.pydata.org/)
* [NumPy](https://numpy.org/)
* [Keras](https://keras.io/)
* [TensorFlow](https://www.tensorflow.org/)
* [Matplotlib](https://matplotlib.org/)
* [SciKit Learn](https://scikit-learn.org/stable/)
* [Seaborn](https://seaborn.pydata.org/)
* Additionally, to download the sample data, you'll need an account on [Kaggle](https://www.kaggle.com/).

### Other stuff that might be interesting:
* [Jupyter](https://jupyter.org/)
* [Anaconda](https://anaconda.org/)
* [Google Colab](https://colab.research.google.com/)
* [Hugging Face](https://huggingface.co/)

### References:
* [Deep Learning with Python](https://deeplearningwithpython.io/), by Francois Chollet and Matthew Watson. Third Edition free online.
* [Hands-On Machine Learning with Scikit-Learn, Keras, and Tensorflow](https://www.oreilly.com/library/view/hands-on-machine-learning/9781098125967/), 3rd Edition by Aurélien Géron


### Import Libraries
Pandas is an analytics library for Python.

NumPy provides a set of mathematical functions, RNGs, and other neat stuff for numbers.

SciKit-Learn includes a number of different tools for regression.

MatPlotLib is for charts and graphs, and Seaborn does fancier data visualization.

Keras and Tensorflow are the libraries for building neural networks.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import tensorflow as tf

### Read Data into Pandas
This data is available at [Kaggle](https://www.kaggle.com/datasets/chenzhiliang/housing-data?resource=download).

This is a small sample dataset of property values in Singapore. It has 11 columns of mixed string and numeric data. When using this dataset, the `value` column is generally considered the `y` values - it is the value of the property. 

In [ ]:
# download file and update the directory where it lives.
# if using a cloud environment like Colab, refer to documentation for how to access local storage for data
#file_dir = "/Users/mandiwalls/Documents/ShallowEnd"
file_dir = input("Enter the directory where you downloaded the Housing_Data.csv file: ")
df = pd.read_csv(f"{file_dir}/Housing_Data.csv")


### Data investigation and Cleanup
There may be things about the data that we'll need to change. In this case, some of the data is strings or a generic `object` type where we'll want numbers. You may also find rows with no values, rows with incorrect encoding (especially stray unicode characters), and other issues that will prevent the models from working properly.

In [ ]:
# what does the data look like?
df.head()

In [ ]:
# what are the types of the data columns?
df.info()

The columns `floorArea`, `bathrooms`, `bedrooms`, and `pricePerSqFt` are listed as 'object' types, and we'll want those to be numerical if we want to use them in a model. Pandas allows us to convert them. This is done in-place, updating the original dataFrame.

In [ ]:
df['floorArea'] = pd.to_numeric(df['floorArea'], errors='coerce')
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce')
df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
df['pricePerSqFt'] = pd.to_numeric(df['pricePerSqFt'], errors='coerce')

In [ ]:
df.info()

In [ ]:
df.head()

#### Outliers
There's one outlier way out there in crazy land. Pandas can help us drop that.

In [ ]:
plt.scatter(df['floorArea'], df['value'])

In [ ]:
quant = df['floorArea'].quantile(0.999)
df = df[df['floorArea'] < quant]
plt.scatter(df['floorArea'], df['value'])

#### Correlation Matrix
We can use the Seaborn package to generate a nice heatmap from the correlation matrix of this data, which will show us if there are any strong correlations between the inputs and the `value` column.

In [ ]:
corr_matrix = df[['floorArea', 'bathrooms', 'bedrooms', 'value']].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Housing Data Correlation Matrix')
plt.show()

All of our target X-values have moderate correlation to `value`, so we might get some decent predictions from the combined X-value models.

### Basic Methods
Pandas can perform basic statistical calculations on the data in the dataFrame, such as calculating the mean, median, and standard deviation the data in a column.

In [ ]:
average_ppsf = df['pricePerSqFt'].mean()
print(f"Average Price per Square Foot: {average_ppsf}")

### Linear Regression
Let's try a linear regression between the 'value' and 'floorArea' columns.

Here we have to choose an independent variable to be x and a dependent variable to be y. 

I'm pulling out columns from the dataFrame to use as x and y to make things easier. 

The double-brackets `[[]]` pull values only. Instead of columns in a frame, these are now treated as series.

In [ ]:
x = df[['floorArea']]
y = df[['value']]

In [ ]:
x.head()

In [ ]:
x.shape

#### Training and Testing Data Sets
Prep the data for linear regression and prediction. 

Splitting the original dataset into train and test allows for the model to be appraised for effectiveness. 

The training set is used to do just that - train the model. The test data is used to test the effectiveness of the model. This will become more apparent in the neural networks section.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape

In [ ]:
y_train.shape

In [ ]:
model = LinearRegression()
model.fit(x_train, y_train)

In [ ]:
model.score(x_test, y_test)

Based on this regression model, less than a third of the value of the properties can be attributed to the floor area of the property. So the majority of the value of the property is attributable to other characteristics. This also shows how the correlation matrix is useful as a first pass but is not always the best prediction of data relationships.

Let's see where that regression line is.

In [ ]:
# use the model to make a set of predicted y-values for the x-values in the x_test set
y_pred = model.predict(x_test)

# the blue dots represent the original y-values from the y_test set
# the red line is built from the predictions
plt.scatter(x_test, y_test, color='blue', label='Actual data')
plt.plot(x_test, y_pred, color='red', linewidth=2, label='Regression line')
plt.xlabel('floorArea')
plt.ylabel('value')
plt.title('Linear Regression Fit')
plt.legend()
plt.show()

### Logistic Regression
Logistic regression is used when the included data is categorical.  It works best when the output is categorical, but can be used with categorical inputs.

We'll skip this for now, but you can find more in the [docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

### Neural Networks I
When data gets too large or complex, regression models aren't as helpful, and you'll want something more sophisticated.

[Tensorflow](https://tensorflow.org) is an Open Source library for building ML models. They don't have to be huge. You can start with very small models and add more sophisticated operations for more complex use cases. 

Tensorflow allows you to build out neural networks in layers with specific functions and characteristics. There are a lot of options for how to manage inputs, hidden nodes, and outputs. We'll look at a basic neural net here to predict the `value` based on `floorArea` and `bedrooms`.

Let's get another look at our data:

In [ ]:
df.head()

#### Prepare the data

One thing to notice about your raw data when planning to use a neural network, is the orders of magnitude. You can see here that the `value` column is a very large number. The `bedrooms` and `bathrooms` columns are very small numbers. The `pricePerSqFt` column is in the middle, but more small than large. Trying to run all of this data through the network would skew the results, so one of the first exercises to do with this data is to normalize the data columns to a similar scale.

A common method for doing this is using a simple formula based on the mean and the standard error:
`(x - mean) / std_err`

In [ ]:
# copy the interesting columns into a temporary dataset, clean out any missing values, and reindex.
X_data = df[['floorArea', 'bedrooms']].copy()
y_data = df[['value']].copy()

X_data = X_data.dropna()
y_data = y_data.loc[X_data.index]

In [ ]:
# normalize to similar scale
# mean_value = housing_data['value'].mean()
mean_floor = X_data['floorArea'].mean()
mean_beds = X_data['bedrooms'].mean()

# std_value = housing_data['value'].std()
std_floor = X_data['floorArea'].std()
std_beds = X_data['bedrooms'].std()

# housing_data['value'] = (housing_data['value'] - mean_value) / std_value / 10
X_data['floorArea'] = (X_data['floorArea'] - mean_floor) / std_floor
X_data['bedrooms'] = (X_data['bedrooms'] - mean_beds) / std_beds
X_data.head()

Negative numbers here are ok. You can see that the new numbers are of a similar scale.

For the `value` column, we can simply divide by a large enough number to bring the values in line with `bedrooms` and `floorArea`.

In [ ]:
y_data['value'] = y_data['value'] / 1000000.
y_data.head()

Split the dataset into training and testing sets. 80% of the data will be used to train the model, and the remaining 20% will be used to test the model for accuracy.  The `train_test_split` method does this for us easily, choosing a random set each time - I've used a seed here for reproducibility, but you can leave that off and see if/how the model changes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=7)
X_train.info()

#### Build the neural network model.

This uses methods from `keras` to build a three-layer network.
* the first layer is the `Input` layer, which has one argument, the size of the X data rows, or tuples.
* a second, hidden layer that has 3 nodes. This layer will use `relu` in the backpropagation step.
* the final output layer, which requires the size of the Y values. Y can be tuples just like X, especially when working with categorial data. Our Y will only represent the `value` column, so the output is 1 value.

Then compile the model with some arguments on how to track the loss that is generated by the model - the difference between observed Y and predicted Y (sometimes referred to as Y-hat, Y with a ^ above it).

In [ ]:
tf.random.set_seed(70)
input_layer = keras.layers.Input(shape=(2,))
housing_model = keras.models.Sequential([
    keras.layers.Dense(3, activation='relu'),
    keras.layers.Dense(1)
])

housing_model.compile(loss='mae', optimizer='adam', metrics=['mae'])
housing_model.summary()

#### Train the model.

Now run the training data through the model some number of times. Convention uses 20-30 "epochs" to start. Small models may converge much earlier; larger, more complex models may not converge at all and require additional tuning. 

This is a small model with a relatively small dataset, so 30 epochs will likely be more than enough.


In [ ]:
num_epochs = 30
batch_size = 128
history = housing_model.fit(
    X_train, 
    y_train, 
    epochs=num_epochs, 
    batch_size=batch_size, 
    verbose=1, 
    validation_split=0.1
)

#### Investigate the effectiveness of the model

Now we can plot the loss the model produces between the Y values in the dataset and the Y values that were produced by the model. We'll be able to see when the model converges and where the least amount of loss is produced. We can use that for tuning the model, or retraining with new data later; we'll have a better idea of how many epochs this dataset will require to converge.

In [ ]:
print(history.history.keys())

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label="Training Loss")
plt.plot(history.history['val_loss'], label="Validation Loss")
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

Loss in this model is minimized between 10 and 15 epochs.

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['mae'], label="Training Mean Abs Error")
plt.plot(history.history['val_mae'], label="Validation Mean Abs Error")
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training and Validation MAE')
plt.legend()
plt.grid(True)
plt.show()

#### Make Predictions
How close does our model come to the true data?

In [ ]:
X_test.head()

The .predict() method can now be used to gather a new set of Y values from the `X_test` dataset. We'll compare these predicted values to the values we already have in the `y_test` dataset. 

In [ ]:
predictions = housing_model.predict(X_test)

In [ ]:
predictions

In [ ]:
X_test.shape

In [ ]:
predictions.shape

In [ ]:
x_range = pd.Series(np.arange(0,4907,1))

In [ ]:
x_range.shape

In [ ]:
plt.scatter(x_range, y_test, label="True Values", color="blue", marker="o")
plt.scatter(x_range, predictions, label="Predicted Values", color="orange", marker="*")
plt.show()

Our predictions are clumped together, but are within the ranges of values for the `y_test` dataset. We might get better predictions with more data included in our X values.

### Neural Networks II
Let's see if we can get better fit by adding the number of bathrooms each property has to our X data.

In [ ]:
# copy the interesting columns into a temporary dataset, clean out any missing values, and reindex.
X_data = df[['floorArea', 'bedrooms', 'bathrooms']].copy()
y_data = df[['value']].copy()

X_data = X_data.dropna()
y_data = y_data.loc[X_data.index]

In [ ]:
# normalize to similar scale
# mean_value = housing_data['value'].mean()
mean_floor = X_data['floorArea'].mean()
mean_beds = X_data['bedrooms'].mean()
mean_baths = X_data['bathrooms'].mean()

# std_value = housing_data['value'].std()
std_floor = X_data['floorArea'].std()
std_beds = X_data['bedrooms'].std()
std_baths = X_data['bathrooms'].std()

# housing_data['value'] = (housing_data['value'] - mean_value) / std_value / 10
X_data['floorArea'] = (X_data['floorArea'] - mean_floor) / std_floor
X_data['bedrooms'] = (X_data['bedrooms'] - mean_beds) / std_beds
X_data['bathrooms'] = (X_data['bathrooms'] - mean_baths) / std_baths
X_data.head()

In [ ]:
y_data['value'] = y_data['value'] / 1000000.
y_data.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=7)

In [ ]:
X_train.info()

#### Build the neural network
This is the same neural network as the last example, with one important change:

in the `Input` layer, the `shape` of the input is now (3,), to tell the model that our X value tuples will have 3 values.

In [ ]:
tf.random.set_seed(70)
input_layer = keras.layers.Input(shape=(3,))
housing_model = keras.models.Sequential([
    keras.layers.Dense(3, activation='relu'),
    keras.layers.Dense(1)
])

housing_model.compile(loss='mae', optimizer='adam', metrics=['mae'])
housing_model.summary()

In [ ]:
num_epochs = 30
batch_size = 128
history = housing_model.fit(
    X_train, 
    y_train, 
    epochs=num_epochs, 
    batch_size=batch_size, 
    verbose=1, 
    validation_split=0.1
)

In [ ]:
print(history.history.keys())

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label="Training Loss")
plt.plot(history.history['val_loss'], label="Validation Loss")
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

This version of the model appears to minimize loss later, at epoch 22. 

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['mae'], label="Training Mean Abs Error")
plt.plot(history.history['val_mae'], label="Validation Mean Abs Error")
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training and Validation MAE')
plt.legend()
plt.grid(True)
plt.show()

### Make Predictions

How close does my model get to the true values?

In [ ]:
X_test.head()

In [ ]:
predictions = housing_model.predict(X_test)

In [ ]:
predictions

In [ ]:
X_test.shape

In [ ]:
y_test.shape

In [ ]:
predictions.shape

In [ ]:
import numpy as np
x_range = pd.Series(np.arange(0,4769,1))

In [ ]:
x_range.shape

In [ ]:
plt.scatter(x_range, y_test, label="True Values", color="blue", marker="o")
plt.scatter(x_range, predictions, label="Predicted Values", color="orange", marker="*")
plt.show()

Our predicted values have a wider range across the range of values of Y. You also might see a difference in the Y-values from the first network, since the data captured in `train_test_split` would be different.

### What would I do next?
This model can be exported and shared to other people on my team. They can use this model to predict the value of new properties based on the number of bedrooms and floorArea of the new property.

When my model is no longer suitable - the circumstances of the envionment have changed to where it is no longer reliably predicting values - the model can be reloaded and retrained on new data to update the weights.

In [ ]:
housing_model.save(f"{file_dir}/housing_model.keras")